# LangChain: Evaluation (Groq Llama 3.1)

## Outline
* **Example Generation** - Create test data (manual + LLM-generated)
* **Manual Evaluation** - Debug mode to inspect chain internals
* **LLM-Assisted Evaluation** - Use LLM to grade responses
* **Evaluation Platform** - Track and visualize results

**Model Used:** Groq Llama-3.1-8b-instant  
**Key Concept:** Evaluating LLM applications is challenging because outputs are open-ended text, not exact matches.

**Why This Matters:** You need to know if your changes improve or degrade your application's performance.


In [ ]:
# Cell 2: Install Required Packages

!pip install langchain langchain-groq python-dotenv docarray sentence-transformers -q

In [ ]:
# Cell 3: Import Libraries and Load Environment

import os
import warnings
from dotenv import load_dotenv, find_dotenv

# Load environment variables
load_dotenv(find_dotenv())

# Suppress warnings
warnings.filterwarnings('ignore')

# Verify API key
groq_api_key = os.environ.get('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file!")
    
print("✅ Environment loaded successfully")


In [ ]:
# Cell 4: Create Q&A Application

from langchain.chains import RetrievalQA
from langchain_groq import ChatGroq
from langchain.document_loaders import CSVLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load data
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)
data = loader.load()

print(f"✅ Loaded {len(data)} documents from {file}")


In [ ]:
# Cell 5: Create Vector Store Index

# Use HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create index
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embeddings
).from_loaders([loader])

print("✅ Vector store index created")


In [ ]:
# Cell 6: Initialize LLM and QA Chain

# Initialize Groq LLM
llm_model = "llama-3.1-8b-instant"
llm = ChatGroq(temperature=0.0, model=llm_model, groq_api_key=groq_api_key)

# Create RetrievalQA chain
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=index.vectorstore.as_retriever(), 
    verbose=True,
    chain_type_kwargs={
        "document_separator": "<<<<>>>>>"
    }
)

print("✅ QA chain created successfully")


In [ ]:
# Cell 7: Inspect Sample Documents

# Look at some sample documents to understand the data
print("Document 10:")
print(data[10])
print("\n" + "="*60 + "\n")

print("Document 11:")
print(data[11])


## Coming Up With Test Data

We need examples to evaluate our QA system. There are two approaches:
1. **Manual** - Create examples by hand (time-consuming but precise)
2. **LLM-Generated** - Use an LLM to generate examples (fast but needs review)


In [ ]:
# Cell 8: Create Hard-Coded Examples

# Manually created question-answer pairs based on the documents above
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

print(f"✅ Created {len(examples)} manual examples")


In [ ]:
# Cell 9: LLM-Generated Examples

from langchain.evaluation.qa import QAGenerateChain

# Create a chain that generates Q&A pairs from documents
example_gen_chain = QAGenerateChain.from_llm(
    ChatGroq(model=llm_model, groq_api_key=groq_api_key)
)

print("✅ QA generation chain created")


In [ ]:
# Cell 10: Generate Examples from Documents

# Generate Q&A pairs from first 5 documents
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

print(f"✅ Generated {len(new_examples)} new examples")
print("\nFirst generated example:")
print(new_examples[0])


In [ ]:
# Cell 11: Compare Generated Example with Source

print("Generated Question:")
print(new_examples[0]['query'])
print("\nGenerated Answer:")
print(new_examples[0]['answer'])
print("\n" + "="*60)
print("\nSource Document:")
print(data[0])


In [ ]:
# Cell 12: Combine All Examples

# Combine manual and LLM-generated examples
examples += new_examples

print(f"✅ Total examples: {len(examples)}")
print(f"   - Manual: 2")
print(f"   - LLM-generated: {len(new_examples)}")


In [ ]:
# Cell 13: Test QA Chain on First Example

# Run the first example through the chain
result = qa.run(examples[0]["query"])

print("Query:")
print(examples[0]["query"])
print("\nExpected Answer:")
print(examples[0]["answer"])
print("\nActual Answer:")
print(result)


## Manual Evaluation

To understand what's happening inside the chain, we can enable **debug mode**. This shows:
- The exact prompt sent to the LLM
- Retrieved documents used as context
- Intermediate steps in multi-step chains
- Token usage and costs


In [ ]:
# Cell 14: Enable Debug Mode

import langchain
langchain.debug = True

print("✅ Debug mode enabled")


In [ ]:
# Cell 15: Run Example with Debug Output

# Run the same example with debug mode on
result = qa.run(examples[0]["query"])

print("\n" + "="*60)
print("Final Answer:")
print(result)


In [ ]:
# Cell 16: Disable Debug Mode

# Turn off debug mode for cleaner output
langchain.debug = False

print("✅ Debug mode disabled")


In [ ]:
# Cell 17: Generate Predictions for All Examples

# Run QA chain on all examples to get predictions
predictions = qa.apply(examples)

print(f"✅ Generated predictions for {len(predictions)} examples")


## LLM-Assisted Evaluation

**The Challenge:** How do we automatically evaluate if answers are correct?
- Can't use exact string matching (too strict)
- Answers can be phrased differently but mean the same thing
- Need semantic comparison

**The Solution:** Use an LLM to grade the answers!


In [ ]:
# Cell 18: Create Evaluation Chain

from langchain.evaluation.qa import QAEvalChain

# Create evaluation chain
eval_chain = QAEvalChain.from_llm(
    ChatGroq(temperature=0, model=llm_model, groq_api_key=groq_api_key)
)

print("✅ Evaluation chain created")


In [ ]:
# Cell 19: Evaluate Predictions

# Use LLM to grade the predictions
graded_outputs = eval_chain.evaluate(examples, predictions)

print("✅ Evaluation complete")


In [ ]:
# Cell 21: Inspect First Graded Output

print("Detailed view of first graded output:")
print(graded_outputs[0])


## Why LLM-Based Evaluation Works

Let's examine Example 0:
- **Question:** "Does the Cozy Comfort Pullover Set have side pockets?"
- **Expected:** "Yes"
- **Predicted:** "The Cozy Comfort Pullover Set, Stripe does have side pockets."

### String Matching Would Fail:
- Expected and predicted strings are completely different
- "Yes" doesn't appear in the predicted answer
- No regex would catch this

### LLM Evaluation Succeeds:
- Understands semantic meaning
- Recognizes both answers convey the same information
- Grades as CORRECT

This is why traditional evaluation metrics fail for LLMs and why we need LLM-assisted evaluation.


In [ ]:
# Cell 23: Count Correct Predictions

# Count how many predictions were graded as correct
correct_count = sum(
    1 for output in graded_outputs 
    if 'CORRECT' in output['text'].upper()
)

print(f"Evaluation Summary:")
print(f"  Total examples: {len(examples)}")
print(f"  Correct predictions: {correct_count}")
print(f"  Accuracy: {correct_count/len(examples)*100:.1f}%")


In [ ]:
## LangChain Evaluation Platform

While this notebook shows evaluation in code, LangChain also provides a **UI-based evaluation platform** that:

### Features:
- **Persistent Sessions** - Track all runs over time
- **Visual Debugging** - See chain steps in a tree view
- **Dataset Creation** - Save examples for future evaluation
- **Comparison** - Compare different chain configurations
- **Collaboration** - Share results with team

### How to Use:
1. Sign up at https://smith.langchain.com/
2. Add `LANGCHAIN_TRACING_V2=true` to your .env
3. Add your API key to .env
4. Run chains normally - they'll auto-log to the platform

### Benefits Over Notebook:
- Historical tracking
- Better visualization
- Team collaboration
- Production monitoring


In [ ]:
# Cell 25: Export Results for Analysis

import json

# Create evaluation report
evaluation_report = {
    "model": llm_model,
    "total_examples": len(examples),
    "predictions": [
        {
            "question": pred['query'],
            "expected": pred['answer'],
            "predicted": pred['result'],
            "grade": graded_outputs[i]['text']
        }
        for i, pred in enumerate(predictions)
    ]
}

# Save to file
with open('evaluation_report.json', 'w') as f:
    json.dump(evaluation_report, f, indent=2)

print("✅ Evaluation report saved to 'evaluation_report.json'")


## Summary: Evaluation Best Practices

### 1. **Create Diverse Test Sets**
- Mix manual and LLM-generated examples
- Cover edge cases and common queries
- Include expected failures

### 2. **Use Debug Mode During Development**
- Understand what's being passed to the LLM
- Check retrieved documents for relevance
- Monitor token usage for cost control

### 3. **Automate Evaluation with LLMs**
- String matching fails for open-ended generation
- LLMs can judge semantic similarity
- Fast iteration on improvements

### 4. **Track Over Time**
- Save evaluation results
- Compare before/after changes
- Use evaluation platform for production monitoring

### Common Pitfalls:
❌ Not enough test examples (need 20-100+)  
❌ Only testing happy path  
❌ Ignoring retrieval quality (checking only final answer)  
❌ Not tracking changes over time  

### Evaluation Workflow:
- Create examples (manual + LLM)
- Run predictions through chain
- Grade with LLM evaluator
- Debug failures with langchain.debug=True
- Iterate and improve
- Repeat